In [1]:
# Import required library
import pandas as pd

In [11]:
# Load dataset from your path
df = pd.read_csv(r"C:\Users\JIIC\Downloads\dirty_financial_transactions.csv")

# Preview first rows
df.head(10)

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed
5,T0006,2021-10-26,C4241,Smartphone,598.0,NaN,PayPal,Completed
6,NaN,2025-02-30,C1313,Laptop,10.0,NaN,credit card,Completed
7,T0008,2023-13-01,C4736,Headphones,669.0,-86.92126929493884,Cash,NaN
8,T0009,NaN,C3387,Tablet,10.0,461.70198437439694,PayPal,NaN
9,T0010,2025-02-30,C2846,Laptop,-1.0,404.8907066405689,creditcard,Pending


In [3]:
# Check structure of dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Transaction_ID      94982 non-null   str    
 1   Transaction_Date    95120 non-null   str    
 2   Customer_ID         95122 non-null   str    
 3   Product_Name        100000 non-null  str    
 4   Quantity            94981 non-null   float64
 5   Price               66503 non-null   str    
 6   Payment_Method      100000 non-null  str    
 7   Transaction_Status  83321 non-null   str    
dtypes: float64(1), str(7)
memory usage: 6.1 MB


In [4]:
df.columns

Index(['Transaction_ID', 'Transaction_Date', 'Customer_ID', 'Product_Name',
       'Quantity', 'Price', 'Payment_Method', 'Transaction_Status'],
      dtype='str')

In [19]:
# Clean column names: lowercase + remove spaces
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Check again
df.columns

Index(['transaction_id', 'transaction_date', 'customer_id', 'product_name',
       'quantity', 'price', 'payment_method', 'transaction_status'],
      dtype='str')

In [6]:
# Remove $ and commas, convert to numeric
df['price'] = pd.to_numeric(df['price'].replace('[$,]', '', regex=True), errors='coerce')

# Check result
df['price'].head()

0    420.210000
1   -445.342025
2    810.993012
3    868.608341
4   -763.122449
Name: price, dtype: float64

In [7]:
# Ensure quantity is numeric
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

# Convert to positive
df['quantity'] = df['quantity'].abs()

df['quantity'].head()

0      5.0
1    469.0
2      4.0
3      7.0
4     10.0
Name: quantity, dtype: float64

In [8]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce') 
df['transaction_date'].head()

0   2024-08-02
1   2020-02-10
2          NaT
3   2020-08-17
4          NaT
Name: transaction_date, dtype: datetime64[us]

In [10]:
# Standardize text columns
df['payment_method'] = df['payment_method'].str.lower().str.strip()
df['transaction_status'] = df['transaction_status'].str.lower().str.strip()
df['product_name'] = df['product_name'].str.lower().str.strip()

In [9]:
# Count invalid dates
df['transaction_date'].isnull().sum()

np.int64(68261)

In [13]:
# Remove duplicate rows
df = df.drop_duplicates()

In [17]:
df['Transaction_Status'].unique()

<StringArray>
[nan, 'Pending', 'completed', 'Completed', 'complete', 'Failed']
Length: 6, dtype: str

In [23]:
df['product_name'].unique()

<StringArray>
[    'Headphones',        'Coffee ',         'Tablet',            'Tab',
 'Coffee Machine',     'Smartphone',         'Laptop',      'Coffee Ma',
            'Cof',           'Smar',       'Coffee M',              'T',
         'Smartp',          'Headp',          'Smart',             'La',
           'Lapt',           'Tabl',              'L',              'C',
        'Smartph',            'Hea',           'Head',       'Smartpho',
          'Lapto',       'Headphon',          'Table',             'Co',
      'Headphone',     'Coffee Mac',             'Sm',         'Coffee',
         'Headph',              'S',    'Coffee Mach',      'Smartphon',
        'Headpho',  'Coffee Machin',           'Coff',            'Lap',
              'H',             'He',             'Ta',   'Coffee Machi',
          'Coffe',            'Sma']
Length: 46, dtype: str

In [24]:
df['payment_method'].unique()

<StringArray>
[    'pay pal',  'creditcard', 'credit card',      'PayPal',        'Cash',
     'PayPal ', 'Credit Card']
Length: 7, dtype: str

In [26]:
# Standardize transaction status
df['transaction_status'] = df['transaction_status'].str.lower().str.strip()

# Fix inconsistent values
df['transaction_status'] = df['transaction_status'].replace({
    'complete': 'completed'
})

# Fill missing values
df['transaction_status'] = df['transaction_status'].fillna('unknown')

# Check result
df['transaction_status'].value_counts(dropna=False)

transaction_status
completed    49413
failed       16629
unknown      16521
pending      16443
Name: count, dtype: int64

In [27]:
# Standardize payment method
df['payment_method'] = df['payment_method'].str.lower().str.strip()

# Fix inconsistencies
df['payment_method'] = df['payment_method'].replace({
    'pay pal': 'paypal',
    'paypal ': 'paypal',
    'creditcard': 'credit card',
    'credit card': 'credit card'
})

# Check result
df['payment_method'].value_counts()

payment_method
credit card    42604
paypal         42349
cash           14053
Name: count, dtype: int64

In [30]:
# Standardize text
df['product_name'] = df['product_name'].str.lower().str.strip()

# Replace inconsistent values
df['product_name'] = df['product_name'].replace({
    'tab': 'tablet',
    'tabl': 'tablet',
    'table': 'tablet',
    't': 'tablet',
    'ta': 'tablet',

    'smart': 'smartphone',
    'smar': 'smartphone',
    's': 'smartphone',
    'smartp': 'smartphone',
    'smartph': 'smartphone',
    'sma': 'smartphone',
    'sm': 'smartphone',
    'smartpho': 'smartphone',
    'smartphon': 'smartphone',

    'lap': 'laptop',
    'lapt': 'laptop',
    'lapto': 'laptop',
    'la': 'laptop',
    'l': 'laptop',

    'head': 'headphones',
    'hea': 'headphones',
    'headp': 'headphones',
    'headph': 'headphones',
    'headpho': 'headphones',
    'headphone': 'headphones',
    'he': 'headphones',
    'headphon': 'headphones',
    'h': 'headphones',

    'coffee ': 'coffee',
    'cof': 'coffee',
    'coff': 'coffee',
    'coffe': 'coffee',
    'co': 'coffee',
    'c': 'coffee',

    'coffee ma': 'coffee machine',
    'coffee m': 'coffee machine',
    'coffee mac': 'coffee machine',
    'coffee mach': 'coffee machine',
    'coffee machi': 'coffee machine',
    'coffee machin': 'coffee machine'
})

# Check result
df['product_name'].value_counts()

product_name
tablet            19964
smartphone        19868
laptop            19770
headphones        19731
coffee machine    18766
coffee              907
Name: count, dtype: int64

In [33]:
# Remove $ and commas, convert to numeric
df['price'] = pd.to_numeric(df['price'].replace('[$,]', '', regex=True), errors='coerce')

# Check result
df['price'].head(5)
df['price'].dtype

dtype('float64')

In [34]:
# Count negative prices
(df['price'] < 0).sum()

np.int64(32876)

In [37]:
# Convert to Positive
df['price'] = df['price'].abs()

In [38]:
df['transaction_date'].isnull().sum()

np.int64(4821)

In [39]:
# Percentage of missing values
(df.isnull().sum() / len(df)) * 100

transaction_id         5.015858
transaction_date       4.869402
customer_id            4.878492
product_name           0.000000
quantity               5.021918
price                 33.463628
payment_method         0.000000
transaction_status     0.000000
dtype: float64

In [40]:
# Remove rows missing key identifiers
df = df.dropna(subset=['transaction_id', 'customer_id'])

In [41]:
# Drop rows where price is missing
df = df.dropna(subset=['price'])

In [42]:
df = df.dropna(subset=['quantity'])
df = df.dropna(subset=['transaction_date'])

In [43]:
df.isnull().sum()

transaction_id        0
transaction_date      0
customer_id           0
product_name          0
quantity              0
price                 0
payment_method        0
transaction_status    0
dtype: int64

In [44]:
df['price'].head()

0    420.210000
1    445.342025
2    810.993012
3    868.608341
4    763.122449
Name: price, dtype: float64

In [45]:
# 1. Flag outliers based on thresholds
df['is_outlier'] = (
    (df['quantity'] > 100) |   # quantity too high
    (df['quantity'] < 1)   |   # quantity zero or negative
    (df['price'] > 1000)   |   # price too high
    (df['price'] < 1)          # price too low
)

# 2. Remove outliers (optional)
df_clean = df[~df['is_outlier']].copy()

# 3. Create total amount
df_clean['total_amount'] = df_clean['quantity'] * df_clean['price']

# 4. Check results
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
df_clean[['product_name','quantity','price','total_amount','is_outlier']].head(20)

Original rows: 53790
Cleaned rows: 17847


,product_name,quantity,price,total_amount,is_outlier
10,laptop,10.0,600.839309,6008.393094,False
17,coffee machine,7.0,108.437824,759.064768,False
27,smartphone,7.0,797.340000,5581.380000,False
33,coffee machine,3.0,120.680996,362.042987,False
35,laptop,2.0,596.425764,1192.851528,False
60,headphones,6.0,692.695688,4156.174131,False
74,laptop,6.0,310.011787,1860.070722,False
78,laptop,10.0,523.972855,5239.728550,False
79,coffee machine,4.0,933.723570,3734.894280,False
80,smartphone,8.0,845.030640,6760.245117,False


In [46]:
# Define thresholds per product
thresholds = {
    'coffee': {'max_quantity': 1000, 'max_price': 100},
    'headphones': {'max_quantity': 100, 'max_price': 1000},
    'laptop': {'max_quantity': 50, 'max_price': 2000},
    'tablet': {'max_quantity': 50, 'max_price': 1500},
    'smartphone': {'max_quantity': 100, 'max_price': 1500},
    'coffee machine': {'max_quantity': 100, 'max_price': 500}
}

# Flag outliers dynamically
def flag_outlier(row):
    prod = row['product_name']
    if prod in thresholds:
        return (
            row['quantity'] > thresholds[prod]['max_quantity'] or
            row['quantity'] < 1 or
            row['price'] > thresholds[prod]['max_price'] or
            row['price'] < 1
        )
    else:
        return True  # Unknown products → remove

df['is_outlier'] = df.apply(flag_outlier, axis=1)
df_clean = df[~df['is_outlier']].copy()
df_clean['total_amount'] = df_clean['quantity'] * df_clean['price']

print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))

Original rows: 53790
Cleaned rows: 15872


In [50]:
# Reset index and drop the old one
df_clean = df_clean.reset_index(drop=True)

# Check first 5 rows
df_clean.head()

,transaction_id,transaction_date,customer_id,product_name,quantity,price,payment_method,transaction_status,is_outlier,total_amount
0,T0011,2023-13-01,C1004,laptop,10.0,600.839309,paypal,completed,False,6008.393094
1,T0018,2025-02-30,C847,coffee machine,7.0,108.437824,credit card,completed,False,759.064768
2,T0028,2025-02-30,C3025,smartphone,7.0,797.340000,paypal,completed,False,5581.380000
3,T0034,2025-02-30,C1406,coffee machine,3.0,120.680996,credit card,completed,False,362.042987
4,T0036,2023-13-01,C292,laptop,2.0,596.425764,paypal,completed,False,1192.851528


In [51]:
df_clean.to_csv(r"C:\Users\JIIC\Downloads\cleaned_financial_transactions.csv", index=False)